In [1]:
import sys, os
sys.path.insert(0, '../../utils')

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import MultipleLocator, PercentFormatter
from utils import load_neurons_table, load_synapses_position_transformed, load_neuron_position_transformed
from neuron_custom_features import calc_basic_degrree_features
from connectome_types import DATA_BASE_PATH
from neuron_custom_features import calc_spines_features
from spines_utils import filter_valid_neuron_w_spines
from plot_utils import ex_color, inh_color
from stats_corr import add_reg_line, add_log_curve
from spine_pref_utils import per_neuron_spine_ratio, plot_spine_along_axon
from figures_utils import add_panel_label, plot_nested_cylinders
from matplotlib.lines import Line2D

In [ ]:
_APN = os.path.join(DATA_BASE_PATH, 'all_axon_pr_network')
CONNECTOME_SYN_TABLE_PATH = os.path.join(_APN, 'connectome_synapses.csv')
CONNECTOME_PRE_SYN_TABLE_PATH = os.path.join(_APN, 'connectome_outgoing_synapses.csv')
SPINE_TABLE_OUTGOING = os.path.join(_APN, 'spine_table_outgoing.csv')
_SPINE_TABLE = os.path.join(_APN, 'spine_table.csv')
_NEURON_TABLE = os.path.join(_APN, 'connectome_neurons.csv')
CONNECTIVITY_DIR    = os.path.join(_APN, 'connectivity_matrix')

neurons_df = load_neuron_position_transformed(neuron_table=_NEURON_TABLE)
calc_basic_degrree_features(neurons_df)
syn_df = load_synapses_position_transformed(base_syn_table_path=CONNECTOME_SYN_TABLE_PATH)
df, syn_with_tags = calc_spines_features(neurons_df, syn_df, spine_table=_SPINE_TABLE, spine_table_outgoing=SPINE_TABLE_OUTGOING)
neuron_clf_type = df[['root_id', 'clf_type']].set_index('root_id').to_dict(orient='index')
df, filtered_syn_mat, filtered_bin_mat, filtered_mapping, filtered_reverse_mapping, ex_neurons, inh_neurons = filter_valid_neuron_w_spines(df, connectivity_dir=CONNECTIVITY_DIR)

spine_df_outgoing = pd.read_csv(SPINE_TABLE_OUTGOING)
outgoing_syn_df = load_synapses_position_transformed(base_syn_table_path=CONNECTOME_PRE_SYN_TABLE_PATH)
outgoing_syn_with_tags = outgoing_syn_df[outgoing_syn_df.id_.isin(spine_df_outgoing.target_id)].copy()
outgoing_syn_with_tags['tag'] = outgoing_syn_with_tags.id_.map(spine_df_outgoing.set_index('target_id').tag)
print(f'neurons post: {outgoing_syn_with_tags.post_id.nunique()}')
print(f'neurons pre: {outgoing_syn_with_tags.pre_id.nunique()}')
print(f'outgoing synapses with tags: {outgoing_syn_with_tags.shape[0]}')

ex_outgoing_syn_with_tags = outgoing_syn_with_tags[outgoing_syn_with_tags.pre_clf_type == 'E']
inh_outgoing_syn_with_tags = outgoing_syn_with_tags[outgoing_syn_with_tags.pre_clf_type == 'I']
print(f'ex outgoing synapses with tags: {ex_outgoing_syn_with_tags.shape[0]}')
print(f'inh outgoing synapses with tags: {inh_outgoing_syn_with_tags.shape[0]}')

ex_neurons  = ex_neurons.copy()
inh_neurons = inh_neurons.copy()

connectome neurons table:  2192
valid neurons w position: 1325
spine table incoming size (8204680, 4)
spine table outgoing size (1979674, 4)
neurons: 2190
synapses with tags: 354258


100%|██████████| 2192/2192 [00:44<00:00, 49.12it/s]


Filtering neurons with valid spine data...
Remaining neurons after filtering: 1994
fixing networks
Filtering: Reducing matrix from 2192 to 1994 neurons.


In [ ]:
main_feature = 'spine'

e_all_outside_stats = per_neuron_spine_ratio(spine_df_outgoing, ex_neurons.root_id.tolist(), group_by='pre_pt_root_id')
i_all_outside_stats = per_neuron_spine_ratio(spine_df_outgoing, inh_neurons.root_id.tolist(), group_by='pre_pt_root_id')

ex_neurons['x_all_outside'] = ex_neurons.root_id.map(e_all_outside_stats['n_syn'])
inh_neurons['x_all_outside'] = inh_neurons.root_id.map(i_all_outside_stats['n_syn'])

ex_neurons[f'outgoing_{main_feature}_ratio_all_outside'] = ex_neurons.root_id.map(e_all_outside_stats['ratio'])
inh_neurons[f'outgoing_{main_feature}_ratio_all_outside'] = inh_neurons.root_id.map(i_all_outside_stats['ratio'])

In [ ]:
## filter only fully proof-read

pr_df = pd.read_csv(f'{DATA_BASE_PATH}/raw_tables/proofreading_status_and_strategy.csv', index_col=0)
fullax=pr_df[pr_df.strategy_axon == 'axon_fully_extended'].pt_root_id.tolist()

inh_neurons = inh_neurons[inh_neurons.root_id.isin(fullax)]
ex_neurons = ex_neurons[ex_neurons.root_id.isin(fullax)]
print(len(ex_neurons)); print(len(inh_neurons))

In [ ]:
# remove 2 outliers
ex_neurons = ex_neurons[ex_neurons.x_all_outside < 2000]

# remove 2, non E neurons - bad ML classification i belive (they are outside the column)
ex_neurons = ex_neurons[ex_neurons[f'outgoing_{main_feature}_ratio_all_outside'] > 0.3]

In [ ]:
plt.rcParams['font.size'] = 13
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['font.family'] = 'Arial'

scatter_overlay_fontsize = 11

In [ ]:
def plot_legend_EI_clf_type(ax, labels=['E', 'I'], colors=[ex_color, inh_color],
                             fontsize='small', markersize=6):
    legend_elements = [
        Line2D([0], [0], marker='o', color=colors[i], markerfacecolor=colors[i],
               linestyle='None', markersize=markersize, label=labels[i])
        for i in range(len(labels))
    ]
    ax.legend(handles=legend_elements, frameon=False, title='',
              loc='lower center', bbox_to_anchor=(0.5, 1.1), ncol=2, fontsize=fontsize)
    ax.set_axis_off()

In [ ]:
fig = plt.figure(figsize=(18, 7), dpi=600)
gs_master = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.5)

cyl_width = 0.75
wspace_top = 0.5
gs_top = gs_master[0].subgridspec(1, 5, width_ratios=[cyl_width, 1, 1, 1, 1], wspace=wspace_top)
axes_top = []

gs_cyl_top = gs_top[0].subgridspec(2, 1, height_ratios=[0.01, 1], hspace=0.0)
ax_legend_top = fig.add_subplot(gs_cyl_top[0])
axes_top.append(fig.add_subplot(gs_cyl_top[1], projection='3d'))
for i in range(1, 5):
    axes_top.append(fig.add_subplot(gs_top[i]))

plot_nested_cylinders(axes_top[0], show_outer=True, scatter_outside=True, n_dots=75, dot_size=1, ex_color=ex_color, inh_color=inh_color)
plot_legend_EI_clf_type(ax=ax_legend_top, markersize=4)

axes_top[1].set_title('E \u2192 All')
axes_top[1].scatter(x=ex_neurons['axon_length'], y=ex_neurons['x_all_outside'], edgecolors=ex_color,
                    facecolors='none', s=5, alpha=0.7, zorder=5)
add_reg_line(ex_neurons['axon_length'], ex_neurons['x_all_outside'], ax=axes_top[1], color='gray',
             add_reg_text='r_only', reg_text_font_size=scatter_overlay_fontsize, linewidth=1.2)

axes_top[2].scatter(x=ex_neurons['x_all_outside'], y=ex_neurons[f'outgoing_{main_feature}_ratio_all_outside'],
                    edgecolors=ex_color, facecolors='none', s=5, alpha=0.7)
add_log_curve(ex_neurons, 'x_all_outside', f'outgoing_{main_feature}_ratio_all_outside', axes_top[2],
              color='gray', fontsize=scatter_overlay_fontsize, linestyle=':')

axes_top[3].set_title('I \u2192 All')
axes_top[3].scatter(x=inh_neurons['axon_length'], y=inh_neurons['x_all_outside'], edgecolors=inh_color,
                    facecolors='none', s=5, alpha=0.7)
add_reg_line(inh_neurons['axon_length'], inh_neurons['x_all_outside'], ax=axes_top[3], color='gray',
             add_reg_text='r_only', reg_text_font_size=scatter_overlay_fontsize, linewidth=1.2)

axes_top[4].scatter(x=inh_neurons['x_all_outside'], y=inh_neurons[f'outgoing_{main_feature}_ratio_all_outside'],
                    edgecolors=inh_color, facecolors='none', s=5, alpha=0.7)
add_log_curve(inh_neurons, 'x_all_outside', f'outgoing_{main_feature}_ratio_all_outside', axes_top[4],
              color='gray', fontsize=scatter_overlay_fontsize, linestyle=':')

for i, ax in enumerate(axes_top):
    if i in [1, 2, 3, 4]:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    if i in [1, 3]:
        ax.set_xlabel('Total axonal length (\u03bcm)')
        ax.set_ylabel('# of outgoing synapses')
        ax.xaxis.set_minor_locator(MultipleLocator(5000))
    if i in [2, 4]:
        ax.set_xlabel('# of outgoing synapses')
        ax.set_ylabel('% of output synapses\n on target spines')
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0, symbol=''))
    if i == 4:
        ax.xaxis.set_minor_locator(MultipleLocator(2500))

gs_bottom = gs_master[1].subgridspec(1, 5, width_ratios=[cyl_width, 1, 1, 1, 1], wspace=wspace_top)
ax_cyl_bottom = fig.add_subplot(gs_bottom[0], projection='3d')
plot_nested_cylinders(ax_cyl_bottom, show_outer=True, scatter_outside=True, n_dots=75, dot_size=1, ex_color=ex_color, inh_color=inh_color)

axes_bottom_plots = [fig.add_subplot(gs_bottom[1:3]), fig.add_subplot(gs_bottom[3:5])]
axes_bottom_plots[0].set_title('E \u2192 All')
axes_bottom_plots[1].set_title('I \u2192 All')

syn_list = [ex_outgoing_syn_with_tags, inh_outgoing_syn_with_tags]
pop_colors = [ex_color, inh_color]

for idx, (ax, outgoing_syn, pop_color) in enumerate(zip(axes_bottom_plots, syn_list, pop_colors)):
    plot_spine_along_axon(
        ax=ax, outgoing_syn=outgoing_syn, node_id_list=[],
        axon_colors=None, pop_color=pop_color,
        show_xlabel=False, show_legend=False,
        show_cross_distance=(idx == 0),
        cross_dist_textsize=scatter_overlay_fontsize, num_single_bins=80,
        max_distance=1500)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0, symbol=''))
    ax.set_xlabel('Axonal path length from soma (\u03bcm)')
    ax.set_xlim(0, 1500)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes_bottom_plots[1].axhline(0.5, color='gray', linestyle='--', lw=1.5, alpha=0.7)
axes_bottom_plots[0].set_ylabel('% of output synapses\n on target spines')
axes_bottom_plots[1].set_ylabel('')

add_panel_label(axes_top[0], 'A', xy=(-0.0, 1.275))
add_panel_label(axes_top[3], 'B', xy=(-0.05, 1.05))
add_panel_label(ax_cyl_bottom, 'C', xy=(-0.0, 1.275))
add_panel_label(axes_bottom_plots[1], 'D', xy=(-0.03, 1.05))

plt.savefig('fig_s3.pdf', format='pdf', bbox_inches='tight')